In [3]:
import json
import pandas as pd
from drain3 import TemplateMiner
from drain3.template_miner_config import TemplateMinerConfig

# 1. Cấu hình Drain3
config = TemplateMinerConfig()
config.profiling_enabled = False
template_miner = TemplateMiner(config=config)

# Mở rộng từ 08:00 để quét qua mốc Change Point 08:58
start_time = pd.to_datetime('2026-06-01T08:00:00Z')
end_time = pd.to_datetime('2026-06-01T16:00:00Z')

log_events = []

print(f"Đang phân tích timeline log từ {start_time} đến {end_time}...")

# 2. Đọc log và gom vết timestamp cùng với template
with open(CART_LOG_FILE, 'r', encoding='utf-8') as f:
    for line in f:
        log = json.loads(line)
        ts = pd.to_datetime(log['timestamp'])
        
        if start_time <= ts <= end_time:
            message = log.get('message', '')
            level = log.get('level', 'INFO')
            
            full_message = f"[{level}] {message}"
            result = template_miner.add_log_message(full_message)
            template_str = result['template_mined']
            
            # Lưu lại timestamp và cặp template tương ứng
            log_events.append({
                'timestamp': ts,
                'template': template_str,
                'level': level
            })

# 3. Tạo DataFrame để bóc tách thống kê timeline
df_log_timeline = pd.DataFrame(log_events)

# Group theo từng template để tìm: Thời điểm đầu tiên, Thời điểm cuối cùng, và Tổng số lượng
log_timeline_summary = df_log_timeline.groupby('template').agg(
    first_seen=('timestamp', 'min'),
    last_seen=('timestamp', 'max'),
    total_count=('timestamp', 'count'),
    level=('level', 'first')
).reset_index()

# Lọc: Chỉ xem các log hệ thống có tính chất cảnh báo hoặc lỗi (WARN, ERROR)
# Hoặc Minh có thể bỏ filter này nếu muốn xem cả INFO
warn_error_timeline = log_timeline_summary[log_timeline_summary['level'].isin(['WARN', 'ERROR', 'FATAL'])]

# Sắp xếp theo thứ tự thời gian xuất hiện từ sớm nhất đến muộn nhất
warn_error_timeline = warn_error_timeline.sort_values(by='first_seen', ascending=True)

# 4. In kết quả dạng Timeline
print("\n" + "="*45 + " TIMELINE LOG CẢNH BÁO " + "="*45)
print(f"{'First Seen (UTC)':<20} | {'Total Count':<12} | {'Log Pattern Template'}")
print("-" * 115)

for idx, row in warn_error_timeline.iterrows():
    print(f"{row['first_seen'].strftime('%Y-%m-%d %H:%M:%S'):<20} | {row['total_count']:<12d} | {row['template']}")

Đang phân tích timeline log từ 2026-06-01 08:00:00+00:00 đến 2026-06-01 16:00:00+00:00...

============================================= TIMELINE LOG CẢNH BÁO =============================================
First Seen (UTC)     | Total Count  | Log Pattern Template
-------------------------------------------------------------------------------------------------------------------
2026-06-01 08:00:21  | 1            | [WARN] Connection pool nearing limit pool=db connections=46/50
2026-06-01 08:00:44  | 630          | [WARN] ProductCatalogCache eviction failed: heap pressure too high
2026-06-01 08:00:45  | 1            | [WARN] GC overhead limit warning: pause=465ms heap=86%
2026-06-01 08:00:46  | 438          | [WARN] Connection pool nearing limit pool=db <*>
2026-06-01 08:00:49  | 1            | [WARN] Slow response detected endpoint=/api/cart latency=1387ms
2026-06-01 08:01:12  | 413          | [WARN] Slow response detected endpoint=/api/cart <*>
2026-06-01 08:02:10  | 1            | [WA